# QUBO Formulation for Protein Folding on a Lattice
## An Educational Implementation of Irbäck, Knuthson & Mohanty (2025)

**Author:** Jonah Minkoff  
**Date:** 2025  
**Based on:** ["Folding lattice proteins confined on minimal grids using a quantum-inspired encoding"](https://arxiv.org/abs/2510.01890)  
Irbäck, A., Knuthson, L., & Mohanty, S. (2025), *Physical Review E* 112, 045302, arXiv:2510.01890

---

## About This Notebook

This notebook provides a **pedagogical walkthrough** of the QUBO (Quadratic Unconstrained Binary Optimization) formulation for lattice protein folding presented in the paper by Irbäck, Knuthson, and Mohanty (2025).

### Purpose

Rather than implementing a production-scale protein folding solver, this notebook aims to:

1. **Explain** the mathematical formulation step-by-step with detailed derivations
2. **Demonstrate** the approach using **small, tractable toy examples** (4-6 residues)
3. **Validate** correctness through dual calculation methods
4. **Make accessible** the concepts behind quantum optimization for biology

### Why Toy Examples?

The original paper tackles realistic protein folding problems at scale. Here, we use:
- **Small chains** (4-6 amino acids) instead of 10-50+
- **Simple lattices** (4-6 positions) that can be visualized easily
- **Complete walkthroughs** where you can verify calculations by hand
- **Transparent matrices** where every entry has clear meaning

This makes the methodology **understandable** without getting lost in computational complexity.

### What You'll Learn

By the end of this notebook, you'll understand:
- How to encode biological constraints as penalty terms in QUBO
- The process of converting a constrained optimization into an unconstrained one
- How QUBO matrices are constructed component-by-component
- Why this formulation is suitable for quantum annealers and QAOA
- How to validate that your QUBO correctly encodes the original problem

---

## :Original Paper Context

**Key contributions of Irbäck, Knuthson & Mohanty (2025):**
- Formulate lattice protein folding as QUBO using Miyazawa-Jernigan potentials
- Encode three critical constraints: position uniqueness, self-avoidance, and chain connectivity
- Demonstrate the approach on both classical simulated annealing and D-Wave hybrid quantum-classical annealing
- Provide benchmarks on realistic protein sequences (48-residue chains), validated against exhaustive enumeration

**Our contribution:**
- Clean, documented implementation focused on education
- Small examples that illuminate the mathematics
- Comprehensive validation suite
- Step-by-step derivations of all formulas

This notebook is **complementary** to the paper - it provides the foundation to understand their methodology before scaling to production systems.

---

## Table of Contents

1. [Problem Definition](#1-problem-definition) - Understanding the lattice model and constraints
2. [Mathematical Formulation](#2-mathematical-formulation) - Deriving the energy function
3. [QUBO Construction](#3-qubo-construction) - Building the Q matrix component by component
4. [Example: Small Protein Chain](#4-example-small-protein-chain) - HPCH on 2×2 lattice (16 variables)
5. [Validation and Testing](#5-validation-and-testing) - Verifying correctness
6. [Larger Example](#6-larger-example) - HHCHPC on 6-position lattice (36 variables)
7. [Analysis and Discussion](#7-analysis-and-discussion) - Scaling and solver options

## 1. Problem Definition

### The Protein Folding Problem

Proteins naturally settle into low-energy shapes determined by local interactions along the chain. Even in highly simplified settings, predicting these configurations leads to interesting discrete optimization problems.

In this notebook, we work with a classic abstraction: a protein chain embedded on a 2D lattice, where residues interact only with their immediate neighbors on the grid. This model strips away biochemical complexity and focuses on the combinatorial structure we ultimately encode as a QUBO.
### Problem Components

1. **Protein Chain**: A sequence of amino acid residues, each of type:
   - **H** (Hydrophobic)
   - **P** (Polar)  
   - **C** (Charged)

2. **Lattice**: A 2D grid where residues can be placed

3. **Objective**: Minimize the energy by:
   - Maximizing favorable interactions (H-H contacts)
   - Avoiding unfavorable interactions (C-C contacts)
   - Satisfying physical constraints

### Physical Constraints

1. **E1**: Each residue must occupy exactly one lattice position
2. **E2**: Each lattice position can hold at most one residue (self-avoidance)
3. **E3**: Consecutive residues in the chain must be at adjacent lattice positions (connectivity)

### Energy Function

$$E_{\text{total}} = E_{\text{MJ}} + \lambda_1 E_1 + \lambda_2 E_2 + \lambda_3 E_3$$

where $\lambda_1, \lambda_2, \lambda_3$ are penalty weights (Lagrange multipliers).

## 2. Mathematical Formulation

### Binary Variables

We encode the problem using binary variables:

$$b_{i,n} = \begin{cases} 1 & \text{if residue } i \text{ is at position } n \\ 0 & \text{otherwise} \end{cases}$$

For a chain with $N$ residues and $M$ lattice positions, we have $N \times M$ binary variables.

### Energy Terms

#### Illustrative contact interaction

$$E_{\text{MJ}} = -\sum_{|i-j|>1} C(a_i, a_j) \sum_{\langle n,m \rangle} b_{i,n} b_{j,m}$$

where:
- $C(a_i, a_j)$ is the interaction energy between residue types
- $\langle n,m \rangle$ means positions $n$ and $m$ are spatially adjacent
- $|i-j| > 1$ excludes bonded neighbors along the chain

#### Constraint E1: One Position Per Residue

$$E_1 = \sum_i \left(\sum_n b_{i,n} - 1\right)^2$$

Expanding:
$$E_1 = \sum_i \left[-\sum_n b_{i,n} + 2\sum_{n<m} b_{i,n}b_{i,m} + 1\right]$$

#### Constraint E2: One Residue Per Position

$$E_2 = \frac{1}{2}\sum_n \sum_{i \neq j} b_{i,n} b_{j,n} = \sum_n \sum_{i<j} b_{i,n} b_{j,n}$$

#### Constraint E3: Chain Connectivity

$$E_3 = \sum_{i=1}^{N-1} \sum_{n=1}^{M} b_{i,n} \sum_{\substack{m=1 \\ d(n,m)>1}}^{M} b_{i+1,m}$$
 - $d(n,m)$ is the lattice distance between positions $n$ and $m$
 - This penalizes consecutive residues that are NOT at adjacent positions.

## 3. QUBO Construction

### Setup and Imports

We'll use our pre-tested functions for QUBO construction and energy evaluation.

In [ ]:
%%writefile qubo_generation.py
import numpy as np


def validate_adjacency(adj):
    """Require a simple undirected lattice graph with binary adjacency."""
    adj = np.asarray(adj)
    if (adj.ndim != 2 or adj.shape[0] == 0 or adj.shape[0] != adj.shape[1]
            or not np.all((adj == 0) | (adj == 1))
            or not np.array_equal(adj, adj.T)
            or np.any(np.diag(adj))):
        raise ValueError("adjacency must be a nonempty, symmetric, binary matrix with zero diagonal")
    return adj

def bit_index(residue_idx, position_idx, num_positions):
    """Convert (residue, position) to bit index."""
    return residue_idx * num_positions + position_idx

def decode_bit_index(bit_idx, num_positions):
    """Convert bit index to (residue, position)."""
    residue_idx = bit_idx // num_positions
    position_idx = bit_idx % num_positions
    return residue_idx, position_idx

# ============================================================================
# E_MJ: Miyazawa-Jernigan Interaction Energy
# ============================================================================

def build_E_MJ(chain, adj, interaction_matrix):
    """
    Build Q_MJ matrix and polynomial for Miyazawa-Jernigan interactions.
    
    E_MJ = -sum_{|i-j|>1} C(a_i, a_j) * sum_{<n,m>} b_{i,n} * b_{j,m}
    
    Returns:
        Q_MJ: numpy array (num_bits x num_bits)
        polynomial: list of (coefficient, bit_i, bit_j) tuples
        constant: constant offset
    """
    adj = validate_adjacency(adj)
    num_residues = len(chain)
    num_positions = adj.shape[0]
    num_bits = num_residues * num_positions
    
    Q_MJ = np.zeros((num_bits, num_bits))
    polynomial = []
    
    # Iterate over non-consecutive residue pairs
    for i in range(num_residues):
        for j in range(i + 2, num_residues):  # |i-j| > 1
            # Get interaction energy
            if ((chain[i], chain[j]) in interaction_matrix
                    and (chain[j], chain[i]) in interaction_matrix
                    and not np.isclose(interaction_matrix[chain[i], chain[j]],
                                       interaction_matrix[chain[j], chain[i]])):
                raise ValueError("contact energies must be symmetric in residue type")
            interaction = interaction_matrix.get((chain[i], chain[j]),
                                                 interaction_matrix.get((chain[j], chain[i]), 0))
            if interaction == 0:
                continue
            
            # Apply negative sign from E_MJ formula
            coeff = -interaction
            
            # For each pair of adjacent positions
            for n in range(num_positions):
                for m in range(num_positions):
                    if adj[n, m] == 1:  # Positions n and m are adjacent
                        bit_i = bit_index(i, n, num_positions)
                        bit_j = bit_index(j, m, num_positions)
                        
                        # Add to Q matrix (upper triangular)
                        if bit_i < bit_j:
                            Q_MJ[bit_i, bit_j] += coeff
                        else:
                            Q_MJ[bit_j, bit_i] += coeff
                        
                        # Add to polynomial
                        polynomial.append((coeff, bit_i, bit_j))
    
    constant = 0
    return Q_MJ, polynomial, constant


def print_E_MJ_details(chain, adj, interaction_matrix):
    """Print detailed breakdown of E_MJ construction."""
    num_residues = len(chain)
    num_positions = adj.shape[0]
    
    print("="*70)
    print("E_MJ: Miyazawa-Jernigan Interaction Energy")
    print("="*70)
    print(f"Formula: E_MJ = -sum_{{|i-j|>1}} C(a_i, a_j) * sum_{{<n,m>}} b_{{i,n}} * b_{{j,m}}")
    print()
    
    # Find non-consecutive pairs with non-zero interactions
    print("Non-consecutive residue pairs:")
    for i in range(num_residues):
        for j in range(i + 2, num_residues):
            interaction = interaction_matrix.get((chain[i], chain[j]),
                                                 interaction_matrix.get((chain[j], chain[i]), 0))
            print(f"  Pair ({i}, {j}): {chain[i]} and {chain[j]} -> C = {interaction}")
    print()
    
    # Build matrix
    Q_MJ, polynomial, constant = build_E_MJ(chain, adj, interaction_matrix)
    
    print(f"Q_MJ Matrix ({Q_MJ.shape[0]}x{Q_MJ.shape[1]}):")
    print(Q_MJ)
    print()
    
    print(f"Non-zero entries in Q_MJ:")
    for i in range(Q_MJ.shape[0]):
        for j in range(i, Q_MJ.shape[1]):
            if Q_MJ[i, j] != 0:
                res_i, pos_i = decode_bit_index(i, num_positions)
                res_j, pos_j = decode_bit_index(j, num_positions)
                print(f"  Q[{i:2d}, {j:2d}] = {Q_MJ[i,j]:+.1f}  (b_{{{res_i},{pos_i}}} * b_{{{res_j},{pos_j}}})")
    print()
    
    print(f"Polynomial (showing first 10 terms):")
    print("E_MJ = ", end="")
    for idx, (coeff, bit_i, bit_j) in enumerate(polynomial[:10]):
        res_i, pos_i = decode_bit_index(bit_i, num_positions)
        res_j, pos_j = decode_bit_index(bit_j, num_positions)
        if idx > 0:
            print(f"       {coeff:+.1f} * b_{{{res_i},{pos_i}}} * b_{{{res_j},{pos_j}}}")
        else:
            print(f"{coeff:+.1f} * b_{{{res_i},{pos_i}}} * b_{{{res_j},{pos_j}}}")
    if len(polynomial) > 10:
        print(f"       ... ({len(polynomial) - 10} more terms)")
    print()
    
    return Q_MJ, polynomial, constant


# ============================================================================
# E1: One Site Per Amino Acid
# ============================================================================

def build_E1(chain, num_positions):
    """
    Build Q_E1 matrix and polynomial.
    
    E1 = sum_i (sum_n b_{i,n} - 1)^2
       = sum_i [-sum_n b_{i,n} + 2*sum_{n<m} b_{i,n}*b_{i,m} + 1]
    
    Returns:
        Q_E1: numpy array (num_bits x num_bits)
        polynomial: list of (coefficient, bit_i, bit_j) tuples
        constant: constant offset
    """
    if num_positions < 1:
        raise ValueError("num_positions must be positive")
    num_residues = len(chain)
    num_bits = num_residues * num_positions
    
    Q_E1 = np.zeros((num_bits, num_bits))
    polynomial = []
    constant = 0
    
    for i in range(num_residues):
        # Diagonal terms: -1 for each position
        for n in range(num_positions):
            bit_idx = bit_index(i, n, num_positions)
            Q_E1[bit_idx, bit_idx] += -1
            polynomial.append((-1, bit_idx, bit_idx))
        
        # Off-diagonal terms: +2 for each pair of positions within same residue
        for n in range(num_positions):
            for m in range(n + 1, num_positions):
                bit_n = bit_index(i, n, num_positions)
                bit_m = bit_index(i, m, num_positions)
                Q_E1[bit_n, bit_m] += 2
                polynomial.append((2, bit_n, bit_m))
        
        # Constant: +1 per residue
        constant += 1
    
    return Q_E1, polynomial, constant


def print_E1_details(chain, num_positions):
    """Print detailed breakdown of E1 construction."""
    num_residues = len(chain)
    
    print("="*70)
    print("E1: One Site Per Amino Acid")
    print("="*70)
    print(f"Formula: E1 = sum_i (sum_n b_{{i,n}} - 1)^2")
    print()
    
    Q_E1, polynomial, constant = build_E1(chain, num_positions)
    
    print(f"Q_E1 Matrix ({Q_E1.shape[0]}x{Q_E1.shape[1]}):")
    print(Q_E1)
    print()
    
    print(f"Diagonal entries (linear terms):")
    for i in range(Q_E1.shape[0]):
        if Q_E1[i, i] != 0:
            res_i, pos_i = decode_bit_index(i, num_positions)
            print(f"  Q[{i:2d}, {i:2d}] = {Q_E1[i,i]:+.1f}  (b_{{{res_i},{pos_i}}})")
    print()
    
    print(f"Off-diagonal entries (quadratic terms, showing first 10):")
    count = 0
    for i in range(Q_E1.shape[0]):
        for j in range(i+1, Q_E1.shape[1]):
            if Q_E1[i, j] != 0:
                res_i, pos_i = decode_bit_index(i, num_positions)
                res_j, pos_j = decode_bit_index(j, num_positions)
                print(f"  Q[{i:2d}, {j:2d}] = {Q_E1[i,j]:+.1f}  (b_{{{res_i},{pos_i}}} * b_{{{res_j},{pos_j}}})")
                count += 1
                if count >= 10:
                    break
        if count >= 10:
            break
    
    # Count total off-diagonal
    total_off_diag = np.sum(Q_E1 != 0) - num_residues * num_positions
    if total_off_diag > 10:
        print(f"  ... ({total_off_diag - 10} more off-diagonal terms)")
    print()
    
    print(f"Constant: {constant}")
    print()
    
    return Q_E1, polynomial, constant


# ============================================================================
# E2: Self-Avoidance (One Residue Per Position)
# ============================================================================

def build_E2(chain, num_positions):
    """
    Build Q_E2 matrix and polynomial.
    
    E2 = (1/2) * sum_n sum_{i!=j} b_{i,n} * b_{j,n}
       = sum_n sum_{i<j} b_{i,n} * b_{j,n}

    Implementation: We count site_occupancy*(site_occupancy-1) which gives
    the number of ordered pairs, then multiply by 0.5 to get the formula result.
    
    Returns:
        Q_E2: numpy array (num_bits x num_bits)
        polynomial: list of (coefficient, bit_i, bit_j) tuples
        constant: constant offset

        note: since b^2 = b for binary variables, any permutations wont show up at separate or multiplied terms
        Ex. since we have b_{0,1} * b_{1,1}, we'd technically also have b_{1,1} * b_{0,1}, but we'd just combine them, which evaluates to the same thing.
    """
    if num_positions < 1:
        raise ValueError("num_positions must be positive")
    num_residues = len(chain)
    num_bits = num_residues * num_positions
    
    Q_E2 = np.zeros((num_bits, num_bits))
    polynomial = []
    
    # For each position
    for n in range(num_positions):
        # For each pair of residues at that position
        for i in range(num_residues):
            for j in range(i + 1, num_residues):
                bit_i = bit_index(i, n, num_positions)
                bit_j = bit_index(j, n, num_positions)
                
                # Add 1 to Q matrix (using i<j ordering eliminates need for 1/2 factor)
                Q_E2[bit_i, bit_j] += 1

                polynomial.append((1, bit_i, bit_j))
    
    constant = 0
    return Q_E2, polynomial, constant


def print_E2_details(chain, num_positions):
    """Print detailed breakdown of E2 construction."""
    num_residues = len(chain)
    
    print("="*70)
    print("E2: Self-Avoidance (One Residue Per Position)")
    print("="*70)
    print(f"Formula: E2 = (1/2) * sum_n sum_{{i!=j}} b_{{i,n}} * b_{{j,n}}")
    print()
    
    Q_E2, polynomial, constant = build_E2(chain, num_positions)
    
    print(f"Q_E2 Matrix ({Q_E2.shape[0]}x{Q_E2.shape[1]}):")
    print(Q_E2)
    print()
    
    print(f"Off-diagonal entries (showing first 10):")
    count = 0
    for i in range(Q_E2.shape[0]):
        for j in range(i+1, Q_E2.shape[1]):
            if Q_E2[i, j] != 0:
                res_i, pos_i = decode_bit_index(i, num_positions)
                res_j, pos_j = decode_bit_index(j, num_positions)
                print(f"  Q[{i:2d}, {j:2d}] = {Q_E2[i,j]:+.1f}  (b_{{{res_i},{pos_i}}} * b_{{{res_j},{pos_j}}})")
                count += 1
                if count >= 10:
                    break
        if count >= 10:
            break
    
    total_off_diag = np.sum(Q_E2 != 0)
    if total_off_diag > 10:
        print(f"  ... ({total_off_diag - 10} more off-diagonal terms)")
    print()
    
    print(f"Note: All diagonal entries are 0 (E2 is purely quadratic)")
    print(f"Constant: {constant}")
    print()
    
    return Q_E2, polynomial, constant


# ============================================================================
# E3: Chain Connectivity
# ============================================================================

def build_E3(chain, adj):
    """
    Build Q_E3 matrix and polynomial.
    
    E3 = sum_{i=1}^{N-1} sum_n b_{i,n} sum_{||m-n||>1} b_{i+1,m}
    
    This penalizes consecutive residues that are NOT at adjacent positions.
    
    Returns:
        Q_E3: numpy array (num_bits x num_bits)
        polynomial: list of (coefficient, bit_i, bit_j) tuples
        constant: constant offset
    """
    adj = validate_adjacency(adj)
    num_residues = len(chain)
    num_positions = adj.shape[0]
    num_bits = num_residues * num_positions
    
    Q_E3 = np.zeros((num_bits, num_bits))
    polynomial = []
    
    # Build non-adjacency matrix (excluding diagonal)
    non_adj = 1 - adj - np.eye(num_positions)
    
    # For each consecutive residue pair
    for i in range(num_residues - 1):
        # For each pair of non-adjacent positions (n, m)
        for n in range(num_positions):
            for m in range(num_positions):
                if non_adj[n, m] == 1:  # Positions n and m are NOT adjacent
                    bit_i = bit_index(i, n, num_positions)
                    bit_j = bit_index(i + 1, m, num_positions)
                    
                    # Add to Q matrix (upper triangular)
                    if bit_i < bit_j:
                        Q_E3[bit_i, bit_j] += 1
                    else:
                        Q_E3[bit_j, bit_i] += 1
                    
                    # Add to polynomial
                    polynomial.append((1, bit_i, bit_j))
    
    constant = 0
    return Q_E3, polynomial, constant


def print_E3_details(chain, adj):
    """Print detailed breakdown of E3 construction."""
    num_residues = len(chain)
    num_positions = adj.shape[0]
    
    print("="*70)
    print("E3: Chain Connectivity")
    print("="*70)
    print(f"Formula: E3 = sum_{{i=1}}^{{N-1}} sum_n b_{{i,n}} sum_{{||m-n||>1}} b_{{i+1,m}}")
    print()
    print("This penalizes consecutive residues at NON-adjacent positions.")
    print()
    
    # Show non-adjacent position pairs
    non_adj = 1 - adj - np.eye(num_positions)
    print("Non-adjacent position pairs:")
    for n in range(num_positions):
        for m in range(num_positions):
            if non_adj[n, m] == 1:
                print(f"  Positions {n} and {m} are NOT adjacent")
    print()
    
    print(f"Consecutive residue pairs: {num_residues - 1}")
    for i in range(num_residues - 1):
        print(f"  Pair ({i}, {i+1}): {chain[i]} -> {chain[i+1]}")
    print()
    
    Q_E3, polynomial, constant = build_E3(chain, adj)
    
    print(f"Q_E3 Matrix ({Q_E3.shape[0]}x{Q_E3.shape[1]}):")
    print(Q_E3)
    print()
    
    print(f"Non-zero entries in Q_E3:")
    for i in range(Q_E3.shape[0]):
        for j in range(i, Q_E3.shape[1]):
            if Q_E3[i, j] != 0:
                res_i, pos_i = decode_bit_index(i, num_positions)
                res_j, pos_j = decode_bit_index(j, num_positions)
                print(f"  Q[{i:2d}, {j:2d}] = {Q_E3[i,j]:+.1f}  (b_{{{res_i},{pos_i}}} * b_{{{res_j},{pos_j}}})  [res {res_i}->{res_j}, pos {pos_i}->{pos_j} non-adj]")
    print()
    
    print(f"Total terms: {len(polynomial)}")
    print(f"Constant: {constant}")
    print()
    
    return Q_E3, polynomial, constant


# ============================================================================
# Main demonstration
# ============================================================================

if __name__ == "__main__":
    # Example setup
    chain = ['H', 'P', 'C', 'H']
    adj = np.array([[0, 1, 1, 0],
                    [1, 0, 0, 1],
                    [1, 0, 0, 1],
                    [0, 1, 1, 0]])
    
    C = {
        ('H', 'H'): 1, ('C', 'C'): -1,
        ('H', 'C'): 0, ('C', 'H'): 0,
        ('H', 'P'): 0, ('P', 'H'): 0,
        ('C', 'P'): 0, ('P', 'C'): 0,
        ('P', 'P'): 0
    }
    
    num_positions = adj.shape[0]
    
    print("\n" + "="*70)
    print("QUBO FORMULATION FOR PROTEIN FOLDING")
    print("="*70)
    print(f"Chain: {chain}")
    print(f"Number of positions: {num_positions}")
    print(f"Total bits: {len(chain) * num_positions}")
    print()
    print("Position Adjacency Matrix:")
    print(adj)
    print()
    
    # Build and display E_MJ
    Q_MJ, poly_MJ, const_MJ = print_E_MJ_details(chain, adj, C)
    
    # Build and display E1
    Q_E1, poly_E1, const_E1 = print_E1_details(chain, num_positions)
    
    # Build and display E2
    Q_E2, poly_E2, const_E2 = print_E2_details(chain, num_positions)
    
    # Build and display E3
    Q_E3, poly_E3, const_E3 = print_E3_details(chain, adj)
    
    print("="*70)
    print("SUMMARY")
    print("="*70)
    print(f"E_MJ: {len(poly_MJ)} terms, constant = {const_MJ}")
    print(f"E1:   {len(poly_E1)} terms, constant = {const_E1}")
    print(f"E2:   {len(poly_E2)} terms, constant = {const_E2}")
    print(f"E3:   {len(poly_E3)} terms, constant = {const_E3}")
    print()
    
    # Combined totals
    total_terms = len(poly_MJ) + len(poly_E1) + len(poly_E2) + len(poly_E3)
    total_constant = const_MJ + const_E1 + const_E2 + const_E3
    print(f"TOTAL: {total_terms} terms, total constant = {total_constant}")


In [ ]:
%%writefile calc_mods.py
import numpy as np

# === Lagrange Penalties (Globals) ===
L1, L2, L3 = 1.0, 1.0, 1.0  # All set to 1.0 as requested

# === Your original interaction matrix ===
C = {
    ('H', 'H'): 1, ('C', 'C'): -1,
    ('H', 'C'): 0, ('C', 'H'): 0,
    ('H', 'P'): 0, ('P', 'H'): 0,
    ('C', 'P'): 0, ('P', 'C'): 0,
    ('P', 'P'): 0
}

def compute_E_MJ_debug(chain, b, adj, C, verbose=False):
    """
    Compute interaction energy with optional debug output
    """
    N, M = b.shape
    E = 0.0
    contacts = []
    
    for i in range(N):
        for j in range(i+2, N):  # |i-j| > 1 means j >= i+2
            a_i, a_j = chain[i], chain[j]
            if ((a_i, a_j) in C and (a_j, a_i) in C
                    and not np.isclose(C[a_i, a_j], C[a_j, a_i])):
                raise ValueError("contact energies must be symmetric in residue type")
            c_ij = C.get((a_i, a_j), C.get((a_j, a_i), 0.0))
            if c_ij != 0:
                for n in range(M):
                    for m in range(M):
                        if b[i, n] == 1 and b[j, m] == 1 and adj[n, m] == 1:
                            E += c_ij
                            contacts.append((i, j, a_i, a_j, n, m, c_ij))
                            if verbose:
                                print(f"  Contact: amino acids {i}({a_i}) and {j}({a_j}) at sites {n} and {m}, C={c_ij}")
    
    if verbose:
        print(f"  Total interaction sum: {E}")
        print(f"  E_MJ = -{E} = {-E}")
    
    return -E

def compute_E1(b):
    """
    Ensures each amino acid occupies exactly one site
    """
    result = np.sum((np.sum(b, axis=1) - 1) ** 2)
    return int(result)

def compute_E2(b):
    """
    Ensures each site has at most one amino acid
    E2 = (1/2) * sum_n sum_{i≠j} b_{i,n} b_{j,n}
    """
    M = b.shape[1]
    pair_count = 0
    for n in range(M):
        site_occupancy = np.sum(b[:, n])
        pair_count += site_occupancy * (site_occupancy - 1)
    # Each unordered overlap appears twice in the ordered-pair count.
    return int(0.5 * pair_count)

def compute_E3(b, adj):
    """
    Ensures chain connectivity
    """
    N, M = b.shape
    # Create non-adjacency matrix
    non_adj = 1 - adj - np.eye(M)
    
    E3 = 0.0
    for i in range(N - 1):
        E3 += b[i, :] @ non_adj @ b[i+1, :]
    
    return int(E3)

def total_energy(chain, bitstring, adj, C, L1=1.0, L2=1.0, L3=1.0, verbose=False):
    """
    Compute total QUBO energy with optional debug output
    """
    N = len(chain)
    adj = np.asarray(adj)
    M = adj.shape[0]
    if (adj.ndim != 2 or M == 0 or adj.shape != (M, M)
            or not np.all((adj == 0) | (adj == 1))
            or not np.array_equal(adj, adj.T) or np.any(np.diag(adj))):
        raise ValueError("adjacency must be a nonempty, symmetric, binary matrix with zero diagonal")
    if len(bitstring) != N * M or any(bit not in '01' for bit in bitstring):
        raise ValueError(f"bitstring must contain exactly {N * M} binary digits")
    b = np.fromiter((int(bit) for bit in bitstring), dtype=int).reshape(N, M)
    
    if verbose:
        print("\nBit matrix b:")
        print(b)
        print("\nPositions:")
        for i in range(N):
            pos = np.where(b[i, :] == 1)[0]
            print(f"  Amino acid {i} ({chain[i]}): site {pos[0] if len(pos) > 0 else 'none'}")
        print("\nComputing E_MJ:")
    
    E_MJ = compute_E_MJ_debug(chain, b, adj, C, verbose)
    E_1  = compute_E1(b)
    E_2  = compute_E2(b)
    E_3  = compute_E3(b, adj)
    
    total = E_MJ + L1*E_1 + L2*E_2 + L3*E_3
    
    if verbose:
        print(f"\nEnergy components:")
        print(f"  E_MJ = {E_MJ}")
        print(f"  E_1  = {E_1}")
        print(f"  E_2  = {E_2}")
        print(f"  E_3  = {E_3}")
        print(f"\nTotal energy calculation:")
        print(f"  E = {E_MJ} + {L1}×{E_1} + {L2}×{E_2} + {L3}×{E_3}")
        print(f"  E = {E_MJ} + {L1*E_1} + {L2*E_2} + {L3*E_3}")
        print(f"  E = {total}")
    
    return total, (E_MJ, E_1, E_2, E_3)

def is_valid_conformation(breakdown):
    """Check if configuration satisfies all constraints"""
    _, E1, E2, E3 = breakdown
    return E1 == 0 and E2 == 0 and E3 == 0

if __name__ == "__main__":
    # Example usage 1 -> Simple and non-disruptive chain
    print("="*60)
    print("=== Example 1: Valid chain ===")
    print("="*60)
    chain = ['H', 'P', 'C', 'H']
    bitstring = '0010100001000001'
    adj = np.array([[0, 1, 1, 0],
                    [1, 0, 0, 1],
                    [1, 0, 0, 1],
                    [0, 1, 1, 0]])

    total_E, breakdown = total_energy(chain, bitstring, adj, C, L1, L2, L3, verbose=True)
    print(f"\nTotal Energy: {total_E}")
    print(f"Energy Breakdown: MJ={breakdown[0]}, E1={breakdown[1]}, E2={breakdown[2]}, E3={breakdown[3]}")
    print(f"Valid conformation: {is_valid_conformation(breakdown)}")

    # Example usage 2 -> just issues with chain connectivity, elsewise fine
    print("\n" + "="*60)
    print("=== Example 2: Chain connectivity violation ===")
    print("="*60)
    chain = ['H', 'H', 'C', 'H', 'P', 'C']
    bitstring = '100000000010010000001000000100000001'
    adj = np.array([[0, 1, 1, 0, 0, 0],
                    [1, 0, 0, 1, 0, 0],
                    [1, 0, 0, 1, 1, 0],
                    [0, 1, 1, 0, 0, 1],
                    [0, 0, 1, 0, 0, 1],
                    [0, 0, 0, 1, 1, 0]])

    total_E, breakdown = total_energy(chain, bitstring, adj, C, L1, L2, L3, verbose=True)
    print(f"\nTotal Energy: {total_E}")
    print(f"Energy Breakdown: MJ={breakdown[0]}, E1={breakdown[1]}, E2={breakdown[2]}, E3={breakdown[3]}")
    print(f"Valid conformation: {is_valid_conformation(breakdown)}")
    
    # Example usage 3 -> violates all 3 constraints, can't be represented physically
    print("\n" + "="*60)
    print("=== Example 3: Multiple violations ===")
    print("="*60)
    chain = ['H', 'C', 'H', 'P']
    bitstring = '1100010101001010'
    adj = np.array([[0, 1, 1, 0],
                    [1, 0, 0, 1],
                    [1, 0, 0, 1],
                    [0, 1, 1, 0]])

    total_E, breakdown = total_energy(chain, bitstring, adj, C, L1, L2, L3, verbose=True)
    print(f"\nTotal Energy: {total_E}")
    print(f"Energy Breakdown: MJ={breakdown[0]}, E1={breakdown[1]}, E2={breakdown[2]}, E3={breakdown[3]}")
    print(f"Valid conformation: {is_valid_conformation(breakdown)}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import our tested QUBO construction functions
# Note: In Colab, you'll need to upload these files or mount from Drive
from qubo_generation import (
    build_E_MJ,
    build_E1,
    build_E2,
    build_E3,
    bit_index,
    decode_bit_index
)

# Import energy evaluation functions
from calc_mods import (
    compute_E_MJ_debug,
    compute_E1,
    compute_E2,
    compute_E3,
    total_energy,
    is_valid_conformation
)

# Set up plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("✓ Imports successful!")

### Interaction Matrix

We define the pairwise interaction energies between residue types - this tutorial uses an illustrative three-type matrix, not measured Miyazawa-Jernigan coefficients:

In [ ]:
# Interaction matrix (Miyazawa-Jernigan inspired)
C = {
    ('H', 'H'): 1,   # Hydrophobic-Hydrophobic: favorable (positive = lower energy when negated)
    ('C', 'C'): -1,  # Charged-Charged: unfavorable (repulsive)
    ('H', 'C'): 0,   # Neutral
    ('C', 'H'): 0,
    ('H', 'P'): 0,   # Neutral
    ('P', 'H'): 0,
    ('C', 'P'): 0,   # Neutral
    ('P', 'C'): 0,
    ('P', 'P'): 0    # Neutral
}

# Lagrange multipliers (penalty weights)
L1, L2, L3 = 1.0, 1.0, 1.0

print("Interaction Matrix C:")
for key, value in C.items():
    if value != 0:
        print(f"  C{key} = {value:+d}")

print(f"\nLagrange Multipliers: λ1={L1}, λ2={L2}, λ3={L3}")

### Helper Functions for Visualization

In [ ]:
def visualize_qubo_matrix(Q, title="QUBO Matrix", show_values=True):
    """
    Visualize a QUBO matrix as a heatmap.
    """
    plt.figure(figsize=(10, 8))
    if show_values and Q.shape[0] <= 16:
        sns.heatmap(Q, annot=True, fmt='.1f', cmap='RdBu_r', center=0,
                   cbar_kws={'label': 'Coefficient'}, square=True)
    else:
        sns.heatmap(Q, cmap='RdBu_r', center=0,
                   cbar_kws={'label': 'Coefficient'}, square=True)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel('Bit Index', fontsize=12)
    plt.ylabel('Bit Index', fontsize=12)
    plt.tight_layout()
    plt.show()

def print_polynomial(polynomial, constant, energy_name="Energy", max_terms=20):
    """
    Print polynomial in readable format.

    Args:
        polynomial: list of (coefficient, bit_i, bit_j) tuples
        constant: constant offset
        energy_name: name of energy term
        max_terms: maximum number of terms to show
    """
    print(f"\n{'='*70}")
    print(f"{energy_name} Polynomial Form")
    print(f"{'='*70}")

    if constant != 0:
        print(f"\nE = {constant:.1f}", end="")
    else:
        print(f"\nE = ", end="")

    terms_shown = 0
    for coeff, bit_i, bit_j in polynomial[:max_terms]:
        res_i, pos_i = decode_bit_index(bit_i, num_positions)

        if bit_i == bit_j:  # Diagonal (linear term)
            print(f" {coeff:+.1f}·b_{{{res_i},{pos_i}}}", end="")
        else:  # Off-diagonal (quadratic term)
            res_j, pos_j = decode_bit_index(bit_j, num_positions)
            print(f" {coeff:+.1f}·b_{{{res_i},{pos_i}}}·b_{{{res_j},{pos_j}}}", end="")

        terms_shown += 1
        if terms_shown % 3 == 0:  # Line break every 3 terms
            print()
            print("     ", end="")

    if len(polynomial) > max_terms:
        print(f"\n      ... ({len(polynomial) - max_terms} more terms)")

    print(f"\n\nTotal terms: {len(polynomial)}")
    print(f"Constant: {constant}")
    print("="*70)

print("✓ Polynomial printing function defined!")

def decode_bitstring(bitstring, chain, num_positions):
    """
    Decode a bitstring to show which residue is at which position.
    """
    bitstring_clean = bitstring.replace(' ', '')
    assignments = []

    for idx, bit in enumerate(bitstring_clean):
        if bit == '1':
            res_idx, pos_idx = decode_bit_index(idx, num_positions)
            assignments.append((res_idx, chain[res_idx], pos_idx))

    return assignments

def print_configuration(bitstring, chain, num_positions):
    """
    Print the residue-to-position assignments for a bitstring.
    """
    assignments = decode_bitstring(bitstring, chain, num_positions)

    print("Configuration:")
    for res_idx, res_type, pos_idx in sorted(assignments):
        print(f"  Residue {res_idx} ({res_type}) → Position {pos_idx}")

def visualize_lattice_configuration(bitstring, chain, adj, num_positions):
    """
    Visualize the protein configuration on the lattice.
    Shows residues at positions and draws edges between consecutive residues.
    """
    assignments = decode_bitstring(bitstring, chain, num_positions)

    # Create position to residue mapping
    pos_to_res = {}
    res_to_pos = {}  # Track ALL positions for each residue (can be multiple!)
    for res_idx, res_type, pos_idx in assignments:
        if pos_idx in pos_to_res:
            pos_to_res[pos_idx].append((res_idx, res_type))
        else:
            pos_to_res[pos_idx] = [(res_idx, res_type)]

        # Store all positions for this residue
        if res_idx in res_to_pos:
            res_to_pos[res_idx].append(pos_idx)
        else:
            res_to_pos[res_idx] = [pos_idx]

    # Simple 2x2 or 2x3 lattice layout
    n = int(np.sqrt(num_positions))
    if n * n == num_positions:
        layout = 'square'
        rows, cols = n, n
    else:
        layout = 'rect'
        rows = 2
        cols = (num_positions + 1) // 2

    fig, ax = plt.subplots(figsize=(cols*2.5, rows*2.5))

    # Position coordinates
    pos_coords = {}
    for pos in range(num_positions):
        if layout == 'square':
            x, y = pos % cols, pos // cols
        else:
            x, y = pos % cols, pos // cols
        pos_coords[pos] = (x, y)

    # Draw grid background with labeled sites (more visible)
    for pos in range(num_positions):
        x, y = pos_coords[pos]
        # Draw grid square with more visible border
        square = plt.Rectangle((x-0.45, y-0.45), 0.9, 0.9,
                               fill=False, edgecolor='gray',
                               linewidth=2, linestyle='--', alpha=0.5)
        ax.add_patch(square)
        # Add position label in corner
        ax.text(x-0.4, y-0.4, f'{pos}', fontsize=9,
               color='gray', alpha=0.7, fontweight='bold')

    # Draw edges between CONSECUTIVE RESIDUES in the chain
    # Need to draw from ALL positions of residue i to ALL positions of residue i+1
    added_connected = False
    added_broken = False

    for res_idx in range(len(chain) - 1):
        if res_idx in res_to_pos and (res_idx + 1) in res_to_pos:
            positions_current = res_to_pos[res_idx]
            positions_next = res_to_pos[res_idx + 1]

            # Draw edge from each position of current residue to each position of next residue
            for pos1 in positions_current:
                for pos2 in positions_next:
                    x1, y1 = pos_coords[pos1]
                    x2, y2 = pos_coords[pos2]

                    # Calculate edge endpoints (from edge of circle, not center)
                    dx, dy = x2 - x1, y2 - y1
                    dist = np.sqrt(dx**2 + dy**2)
                    if dist > 0:
                        # Offset by circle radius (0.3) in direction of other node
                        offset_x, offset_y = 0.3 * dx / dist, 0.3 * dy / dist
                        x1_edge, y1_edge = x1 + offset_x, y1 + offset_y
                        x2_edge, y2_edge = x2 - offset_x, y2 - offset_y
                    else:
                        x1_edge, y1_edge = x1, y1
                        x2_edge, y2_edge = x2, y2

                    # Check if they're adjacent (connected) or not (broken)
                    if adj[pos1, pos2] == 1:
                        ax.plot([x1_edge, x2_edge], [y1_edge, y2_edge],
                               color='#2ecc71', linewidth=2.5, alpha=0.8,
                               solid_capstyle='round',
                               label='Connected' if not added_connected else '')
                        added_connected = True
                    else:
                        ax.plot([x1_edge, x2_edge], [y1_edge, y2_edge],
                               color='#e74c3c', linewidth=2.5, alpha=0.7,
                               linestyle='--', dashes=(5, 3),
                               label='Broken (E3 violation)' if not added_broken else '')
                        added_broken = True

    # Draw lattice points AFTER edges so circles are on top
    for pos in range(num_positions):
        x, y = pos_coords[pos]

        # Determine color and label
        if pos in pos_to_res:
            residues = pos_to_res[pos]
            label = ', '.join([f"{r[0]}({r[1]})" for r in residues])
            # Use softer colors
            if len(residues) > 1:
                color = '#ff6b6b'  # Soft red for E2 violations
                edge_color = '#c0392b'
                edge_width = 3
            else:
                color = '#74b9ff'  # Soft blue for valid
                edge_color = '#0984e3'
                edge_width = 2
        else:
            label = ''
            color = '#ecf0f1'  # Light gray for empty
            edge_color = '#95a5a6'
            edge_width = 1.5

        circle = plt.Circle((x, y), 0.3, color=color, ec=edge_color,
                           linewidth=edge_width, zorder=10)
        ax.add_patch(circle)

        if label:
            ax.text(x, y, label, ha='center', va='center',
                   fontsize=9, fontweight='bold', zorder=11)

    ax.set_xlim(-0.6, cols-0.4)
    ax.set_ylim(-0.6, rows-0.4)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.invert_yaxis()

    # Add legend in better position (bottom left to avoid blocking nodes)
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(loc='lower left', framealpha=0.95, fontsize=10)

    plt.title('Lattice Configuration (Chain Connectivity)',
             fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("✓ Lattice visualization functions defined!")

## 4. Example: Small Protein Chain

Let's work through a complete example with chain **HPCH** (Hydrophobic-Polar-Charged-Hydrophobic).

### Problem Setup

In [ ]:
# Define the protein chain
chain = ['H', 'P', 'C', 'H']
num_residues = len(chain)
num_positions = 4  # 4 positions on the lattice
num_bits = num_residues * num_positions

# Define lattice adjacency matrix (which positions are neighbors)
# This represents a square lattice:
#   0---1
#   |   |
#   2---3
adj = np.array([[0, 1, 1, 0],
                [1, 0, 0, 1],
                [1, 0, 0, 1],
                [0, 1, 1, 0]])

print(f"Chain: {' → '.join(chain)}")
print(f"Number of residues: {num_residues}")
print(f"Number of positions: {num_positions}")
print(f"Total binary variables: {num_bits}")
print(f"\nPosition Adjacency Matrix:")
print(adj)
print("\nLattice structure:")
print("  0---1")
print("  |   |")
print("  2---3")

## 4a. Building QUBO Components

We'll build each energy term separately using our tested functions.

### E_MJ: Miyazawa-Jernigan Interaction Energy

This term captures the **biological objective** - we want to maximize favorable interactions between non-consecutive residues that end up spatially close.

**Mathematical Form:**

$$E_{\text{MJ}} = -\sum_{\substack{i,j \\ |i-j|>1}} C(a_i, a_j) \sum_{\langle n,m \rangle} b_{i,n} \cdot b_{j,m}$$

where:
- $i, j$ index residues in the chain with $|i-j| > 1$ (non-consecutive)
- $C(a_i, a_j)$ is the interaction energy between residue types $a_i$ and $a_j$
- $\langle n,m \rangle$ denotes spatially adjacent lattice positions
- $b_{i,n} \in \{0,1\}$ indicates whether residue $i$ occupies position $n$

**Key concepts:**
- We only consider residue pairs where $|i-j| > 1$ (non-consecutive in the chain) to avoid double-counting
- The interaction coefficient $C(a_i, a_j)$ represents the strength of interaction:
  - $C(H,H) = +1$: Hydrophobic residues attract (favorable when negated)
  - $C(C,C) = -1$: Charged residues repel (unfavorable)
  - Other pairs are neutral ($C = 0$)
- We check all pairs of adjacent lattice positions where these residues could interact
- The negative sign converts the sum into a minimization problem (lower $E_{\text{MJ}}$ = more favorable interactions)

**QUBO encoding:** Each interaction creates quadratic terms $b_{i,n} \cdot b_{j,m}$ with coefficient $-C(a_i,a_j)$

In [ ]:
# Build E_MJ (Miyazawa-Jernigan interaction)
Q_MJ, poly_MJ, const_MJ = build_E_MJ(chain, adj, C)

print("="*70)
print("E_MJ: Miyazawa-Jernigan Interaction Energy")
print("="*70)
print(f"Matrix shape: {Q_MJ.shape}")
print(f"Number of terms: {len(poly_MJ)}")
print(f"Constant offset: {const_MJ}")
print(f"\nNon-zero entries: {np.count_nonzero(Q_MJ)}")

# Show some non-zero entries
print(f"\nSample non-zero coefficients:")
count = 0
for i in range(Q_MJ.shape[0]):
    for j in range(i, Q_MJ.shape[1]):
        if Q_MJ[i, j] != 0 and count < 5:
            res_i, pos_i = decode_bit_index(i, num_positions)
            res_j, pos_j = decode_bit_index(j, num_positions)
            print(f"  Q[{i:2d},{j:2d}] = {Q_MJ[i,j]:+.1f}  (b_{{{res_i},{pos_i}}} * b_{{{res_j},{pos_j}}})")
            count += 1

visualize_qubo_matrix(Q_MJ, "E_MJ: Interaction Energy Matrix")

In [ ]:
# View E_MJ polynomial
print_polynomial(poly_MJ, const_MJ, "E_MJ: Miyazawa-Jernigan Interaction", max_terms=20)

### E1: One Position Per Residue Constraint

This ensures each residue occupies **exactly one** lattice position - a fundamental physical constraint.

**Mathematical Form:**

$$E_1 = \sum_{i=0}^{N-1} \left(\sum_{n=0}^{M-1} b_{i,n} - 1\right)^2$$

where:
- $N$ is the number of residues
- $M$ is the number of lattice positions
- For each residue $i$, we require $\sum_n b_{i,n} = 1$

**Derivation - Expanding to QUBO form:**

Starting from the squared constraint:

$$\left(\sum_{n=0}^{M-1} b_{i,n} - 1\right)^2 = \left(\sum_n b_{i,n}\right)^2 - 2\sum_n b_{i,n} + 1$$

Expanding the square:

$$= \sum_n b_{i,n}^2 + \sum_{n \neq m} b_{i,n} b_{i,m} - 2\sum_n b_{i,n} + 1$$

Since $b_{i,n}^2 = b_{i,n}$ (binary property):

$$= \sum_n b_{i,n} + 2\sum_{n<m} b_{i,n} b_{i,m} - 2\sum_n b_{i,n} + 1$$

$$= -\sum_n b_{i,n} + 2\sum_{n<m} b_{i,n} b_{i,m} + 1$$

**QUBO encoding:**
- **Diagonal terms:** Coefficient $-1$ for each variable $b_{i,n}$
- **Off-diagonal terms:** Coefficient $+2$ for each pair $(b_{i,n}, b_{i,m})$ with $n < m$ (same residue, different positions)
- **Constant offset:** $+N$ (one for each residue)

In [ ]:
# Build E1 (one position per residue)
Q_E1, poly_E1, const_E1 = build_E1(chain, num_positions)

print("="*70)
print("E1: One Position Per Residue Constraint")
print("="*70)
print(f"Matrix shape: {Q_E1.shape}")
print(f"Number of terms: {len(poly_E1)}")
print(f"Constant offset: {const_E1}")
print(f"\nDiagonal entries (all should be -1): ", np.diag(Q_E1)[:8])

visualize_qubo_matrix(Q_E1, "E1: One Position Per Residue Matrix")

In [ ]:
# View E1 polynomial
print_polynomial(poly_E1, const_E1, "E1: One Position Per Residue", max_terms=20)

### E2: Self-Avoidance Constraint (One Residue Per Position)

This prevents multiple residues from occupying the same lattice position - implementing **spatial exclusion**.

**Mathematical Form:**

$$E_2 = \frac{1}{2}\sum_{n=0}^{M-1} \sum_{\substack{i,j=0 \\ i \neq j}}^{N-1} b_{i,n} \cdot b_{j,n}$$

Equivalently (counting each unordered pair once):

$$E_2 = \sum_{n=0}^{M-1} \sum_{0 \leq i < j < N} b_{i,n} \cdot b_{j,n}$$

where:
- For each position $n$, we penalize any pair of different residues $(i,j)$ both occupying that position
- The factor $\frac{1}{2}$ in the first form accounts for each unordered pair being counted twice
- The second form explicitly sums over $i < j$ to count each pair exactly once

**Physical intuition:** Two residues cannot be at the same place at the same time! If $b_{i,n} = 1$ and $b_{j,n} = 1$ for $i \neq j$, we add a penalty.

**QUBO encoding:**
- **Diagonal terms:** All zero (coefficient $0$) - a residue doesn't collide with itself
- **Off-diagonal terms:** Coefficient $+1.0$ for each pair $(b_{i,n}, b_{j,n})$ with $i < j$ at the same position $n$
- **Constant offset:** $0$

**Example:** If position 2 has residues 0, 1, and 3 all present:
$$E_2 \text{ at position 2} = b_{0,2} \cdot b_{1,2} + b_{0,2} \cdot b_{3,2} + b_{1,2} \cdot b_{3,2} = 1 + 1 + 1 = 3$$

In [ ]:
# Build E2 (one residue per position)
Q_E2, poly_E2, const_E2 = build_E2(chain, num_positions)

print("="*70)
print("E2: One Residue Per Position Constraint (Self-Avoidance)")
print("="*70)
print(f"Matrix shape: {Q_E2.shape}")
print(f"Number of terms: {len(poly_E2)}")
print(f"Constant offset: {const_E2}")
print(f"\nAll diagonal entries are zero: {np.all(np.diag(Q_E2) == 0)}")

visualize_qubo_matrix(Q_E2, "E2: Self-Avoidance Matrix")

In [ ]:
# View E2 polynomial
print_polynomial(poly_E2, const_E2, "E2: Self-Avoidance", max_terms=20)

### E3: Chain Connectivity Constraint

This ensures consecutive residues in the chain are placed at **adjacent lattice positions** - maintaining the physical integrity of the protein backbone.

**Mathematical Form:**

$$E_3 = \sum_{i=0}^{N-2} \sum_{n=0}^{M-1} b_{i,n} \sum_{\substack{m=0 \\ \text{adj}[n,m]=0}}^{M-1} b_{i+1,m}$$

Alternatively written as:

$$E_3 = \sum_{i=1}^{N-1} \sum_{n=1}^{M} b_{i,n} \sum_{\substack{m=1 \\ d(n,m)>1}}^{M} b_{i+1,m}$$

Or from the paper:

$$E_3 = \sum_{i=0}^{N-2} \sum_{n=0}^{M-1} \sum_{\substack{m=0 \\ ||m-n||>1}}^{M-1} b_{i,n} \cdot b_{i+1,m}$$


where:
- We sum over all consecutive residue pairs $(i, i+1)$ in the chain
- For residue $i$ at position $n$ and residue $i+1$ at position $m$
- We add a penalty if positions $n$ and $m$ are **NOT adjacent** (i.e., $\text{adj}[n,m] = 0$)
- $||m-n|| > 1$ indicates non-adjacent positions

**Physical interpretation:**
Residues $i$ and $i+1$ are bonded in the chain, so they must be spatially next to each other on the lattice. If they're placed at non-adjacent positions, the chain is "broken."

**Breaking it down:**
1. If residue $i$ is at position $n$: $b_{i,n} = 1$
2. If residue $i+1$ is at position $m$: $b_{i+1,m} = 1$
3. If $n$ and $m$ are NOT adjacent: add penalty of $1$

**QUBO encoding:**
- **Diagonal terms:** All zero
- **Off-diagonal terms:** Coefficient $+1.0$ for each term $b_{i,n} \cdot b_{i+1,m}$ where positions $n$ and $m$ are non-adjacent
- **Constant offset:** $0$

Valid protein configurations have $E_3 = 0$ (all consecutive residues are at adjacent positions).

In [ ]:
# Build E3 (chain connectivity)
Q_E3, poly_E3, const_E3 = build_E3(chain, adj)

print("="*70)
print("E3: Chain Connectivity Constraint")
print("="*70)
print(f"Matrix shape: {Q_E3.shape}")
print(f"Number of terms: {len(poly_E3)}")
print(f"Constant offset: {const_E3}")

visualize_qubo_matrix(Q_E3, "E3: Chain Connectivity Matrix")

In [ ]:
# View E3 polynomial
print_polynomial(poly_E3, const_E3, "E3: Chain Connectivity", max_terms=20)

### Combining All Terms

Now we combine all four energy components into a single QUBO matrix with Lagrange multipliers:

$$Q_{\text{total}} = Q_{\text{MJ}} + \lambda_1 Q_{E1} + \lambda_2 Q_{E2} + \lambda_3 Q_{E3}$$

$$\text{offset}_{\text{total}} = \text{const}_{\text{MJ}} + \lambda_1 \cdot \text{const}_{E1} + \lambda_2 \cdot \text{const}_{E2} + \lambda_3 \cdot \text{const}_{E3}$$

The complete objective function:

$$E_{\text{total}} = E_{\text{MJ}} + \lambda_1 E_1 + \lambda_2 E_2 + \lambda_3 E_3$$

Where:
- $E_{\text{MJ}}$: **Biological objective** (minimize interaction energy)
- $\lambda_1 E_1$: **Hard constraint** (one position per residue)
- $\lambda_2 E_2$: **Hard constraint** (self-avoidance)
- $\lambda_3 E_3$: **Hard constraint** (chain connectivity)

**Setting $\lambda_1 = \lambda_2 = \lambda_3 = 1.0$** gives equal weight to all constraints. In practice:
- Larger $\lambda_i$ values enforce constraints more strictly
- We want $\lambda_i$ large enough to ensure physically valid solutions
- But not so large that the biological energy becomes negligible

The resulting QUBO can be solved by any optimization method that handles binary variables.

In [ ]:
# Combine all QUBO matrices
Q_total = Q_MJ + L1 * Q_E1 + L2 * Q_E2 + L3 * Q_E3
offset_total = const_MJ + L1 * const_E1 + L2 * const_E2 + L3 * const_E3

print("="*70)
print("Combined QUBO Matrix")
print("="*70)
print(f"Q_total = Q_MJ + {L1}·Q_E1 + {L2}·Q_E2 + {L3}·Q_E3")
print(f"\nMatrix shape: {Q_total.shape}")
print(f"Total constant offset: {offset_total}")
print(f"\nMatrix statistics:")
print(f"  Min value: {Q_total.min():.2f}")
print(f"  Max value: {Q_total.max():.2f}")
print(f"  Non-zero entries: {np.count_nonzero(Q_total)} / {Q_total.size}")
print(f"  Sparsity: {100 * (1 - np.count_nonzero(Q_total) / Q_total.size):.1f}%")

visualize_qubo_matrix(Q_total, "Combined QUBO Matrix")

## 5. Validation and Testing

### Test Case 1: Valid Configuration

Let's test a valid protein folding configuration where all constraints are satisfied.

In [ ]:
# Valid configuration: each residue at exactly one position, no overlaps, chain connected
bitstring_valid = '0010100001000001'

print("="*70)
print("Test Case 1: Valid Configuration")
print("="*70)
print(f"Bitstring: {bitstring_valid}")
print()

# Decode and display configuration
print_configuration(bitstring_valid, chain, num_positions)
print()

# Visualize on lattice
visualize_lattice_configuration(bitstring_valid, chain, adj, num_positions)

# Compute energy using calc_mods
total_E, breakdown = total_energy(chain, bitstring_valid, adj, C, L1, L2, L3, verbose=False)

print(f"\nEnergy Breakdown:")
print(f"  E_MJ (interaction): {breakdown[0]}")
print(f"  E1 (one pos/res):   {breakdown[1]}")
print(f"  E2 (self-avoid):    {breakdown[2]}")
print(f"  E3 (connectivity):  {breakdown[3]}")
print(f"\nTotal Energy: {total_E}")
print(f"Valid conformation: {is_valid_conformation(breakdown)}")

if is_valid_conformation(breakdown):
    print("\n✓ All constraints satisfied! This is a physically valid protein folding.")
else:
    print("\n✗ Constraint violations detected.")

### Test Case 2: Configuration with Violations

Now let's test a configuration that violates multiple constraints.

In [ ]:
# Invalid configuration with multiple violations
chain_test2 = ['H', 'P', 'C', 'H']
bitstring_invalid = '1100010101001010'

print("="*70)
print("Test Case 2: Configuration with Violations")
print("="*70)
print(f"Chain: {' → '.join(chain_test2)}")
print(f"Bitstring: {bitstring_invalid}")
print()

# Decode and display configuration
print_configuration(bitstring_invalid, chain_test2, num_positions)
print()

# Visualize on lattice
visualize_lattice_configuration(bitstring_invalid, chain_test2, adj, num_positions)

# Compute energy
total_E2, breakdown2 = total_energy(chain_test2, bitstring_invalid, adj, C, L1, L2, L3, verbose=False)

print(f"\nEnergy Breakdown:")
print(f"  E_MJ (interaction): {breakdown2[0]}")
print(f"  E1 (one pos/res):   {breakdown2[1]}  {'✗ VIOLATION' if breakdown2[1] > 0 else '✓'}")
print(f"  E2 (self-avoid):    {breakdown2[2]}  {'✗ VIOLATION' if breakdown2[2] > 0 else '✓'}")
print(f"  E3 (connectivity):  {breakdown2[3]}  {'✗ VIOLATION' if breakdown2[3] > 0 else '✓'}")
print(f"\nTotal Energy: {total_E2}")
print(f"Valid conformation: {is_valid_conformation(breakdown2)}")

if not is_valid_conformation(breakdown2):
    print("\n✗ Constraint violations detected:")
    if breakdown2[1] > 0:
        print("  - E1: Some residues occupy multiple positions")
    if breakdown2[2] > 0:
        print("  - E2: Some positions have multiple residues (overlap)")
    if breakdown2[3] > 0:
        print("  - E3: Chain connectivity is broken")

### Verification Against Expected Values

Let's verify our QUBO implementation produces the correct energies by comparing two independent calculation methods.

**Why this verification matters:**

When constructing a QUBO formulation, it's critical to ensure that:
1. The matrix representation correctly encodes the problem
2. Evaluating the QUBO produces the same energy as direct calculation
3. Both methods agree on all test cases (valid and invalid configurations)

**Two methods of energy calculation:**

**Method 1: Direct Calculation** (from `calc_mods.py`)
- Directly implements the mathematical formulas for each energy term
- Iterates over residue pairs, positions, and adjacencies explicitly
- Human-readable and easy to verify by hand
- Formula: $E = E_{\text{MJ}} + \lambda_1 E_1 + \lambda_2 E_2 + \lambda_3 E_3$

**Method 2: QUBO Matrix Evaluation** (from `qubo_generation.py`)
- Encodes the entire problem in a single matrix $Q$ and offset
- Evaluates using: $E = \text{offset} + \sum_i Q_{ii} x_i + \sum_{i<j} Q_{ij} x_i x_j$
- More abstract but necessary for quantum/classical solvers
- This is the form that gets sent to optimization algorithms

**What we're testing:**

For each test configuration (bitstring), we compute:
- Energy via **direct calculation**: Use the explicit formulas
- Energy via **QUBO evaluation**: Use matrix multiplication

If both methods give identical results (within floating-point precision), we can be confident that:
- ✓ The QUBO construction is correct
- ✓ The matrix accurately represents the problem
- ✓ We can trust the QUBO for optimization

**Test cases:**
- **Valid configuration**: All constraints satisfied ($E_1 = E_2 = E_3 = 0$)
- **Invalid configuration**: Multiple violations to test penalty terms

In [ ]:
def evaluate_qubo_energy(bitstring, Q, constant):
    """
    Evaluate QUBO energy for a given bitstring.
    E = constant + sum_i Q[i,i]*x_i + sum_{i<j} Q[i,j]*x_i*x_j
    """
    x = np.array([int(b) for b in bitstring.replace(' ', '')])

    energy = constant

    # Diagonal terms
    for i in range(len(x)):
        energy += Q[i, i] * x[i]

    # Off-diagonal terms
    for i in range(len(x)):
        for j in range(i + 1, len(x)):
            energy += Q[i, j] * x[i] * x[j]

    return energy

# Test on both configurations
print("="*70)
print("QUBO Evaluation Verification")
print("="*70)

# Test 1: Valid configuration
E_qubo_1 = evaluate_qubo_energy(bitstring_valid, Q_total, offset_total)
E_direct_1 = total_E

print(f"\nTest 1 (Valid Config):")
print(f"  Energy via QUBO matrix:  {E_qubo_1:.1f}")
print(f"  Energy via direct calc:  {E_direct_1:.1f}")
print(f"  Match: {abs(E_qubo_1 - E_direct_1) < 0.01}")

# Test 2: Invalid configuration
# Build QUBO for test chain 2
Q_MJ_2, _, const_MJ_2 = build_E_MJ(chain_test2, adj, C)
Q_E1_2, _, const_E1_2 = build_E1(chain_test2, num_positions)
Q_E2_2, _, const_E2_2 = build_E2(chain_test2, num_positions)
Q_E3_2, _, const_E3_2 = build_E3(chain_test2, adj)
Q_total_2 = Q_MJ_2 + L1 * Q_E1_2 + L2 * Q_E2_2 + L3 * Q_E3_2
offset_total_2 = const_MJ_2 + L1 * const_E1_2 + L2 * const_E2_2 + L3 * const_E3_2

E_qubo_2 = evaluate_qubo_energy(bitstring_invalid, Q_total_2, offset_total_2)
E_direct_2 = total_E2

print(f"\nTest 2 (Invalid Config):")
print(f"  Energy via QUBO matrix:  {E_qubo_2:.1f}")
print(f"  Energy via direct calc:  {E_direct_2:.1f}")
print(f"  Match: {abs(E_qubo_2 - E_direct_2) < 0.01}")

if abs(E_qubo_1 - E_direct_1) < 0.01 and abs(E_qubo_2 - E_direct_2) < 0.01:
    print("\n✓ QUBO formulation verified! Both methods produce identical results.")
else:
    print("\n✗ Discrepancy detected between QUBO and direct evaluation.")

## 6. Larger Example

Let's demonstrate the scalability with a larger protein chain: **HHCHPC**

In [ ]:
# Larger example
chain_large = ['H', 'H', 'C', 'H', 'P', 'C']
num_residues_large = len(chain_large)
num_positions_large = 6
num_bits_large = num_residues_large * num_positions_large

# 6-position lattice adjacency
adj_large = np.array([[0, 1, 1, 0, 0, 0],
                      [1, 0, 0, 1, 0, 0],
                      [1, 0, 0, 1, 1, 0],
                      [0, 1, 1, 0, 0, 1],
                      [0, 0, 1, 0, 0, 1],
                      [0, 0, 0, 1, 1, 0]])

print("="*70)
print("Larger Example: HHCHPC")
print("="*70)
print(f"Chain: {' → '.join(chain_large)}")
print(f"Number of residues: {num_residues_large}")
print(f"Number of positions: {num_positions_large}")
print(f"Total binary variables: {num_bits_large}")
print(f"\nPosition Adjacency Matrix:")
print(adj_large)

In [ ]:
# Build QUBO for larger example
Q_MJ_large, poly_MJ_large, const_MJ_large = build_E_MJ(chain_large, adj_large, C)
Q_E1_large, poly_E1_large, const_E1_large = build_E1(chain_large, num_positions_large)
Q_E2_large, poly_E2_large, const_E2_large = build_E2(chain_large, num_positions_large)
Q_E3_large, poly_E3_large, const_E3_large = build_E3(chain_large, adj_large)

Q_total_large = Q_MJ_large + L1 * Q_E1_large + L2 * Q_E2_large + L3 * Q_E3_large
offset_total_large = const_MJ_large + L1 * const_E1_large + L2 * const_E2_large + L3 * const_E3_large

print("="*70)
print("QUBO Statistics for Larger Example")
print("="*70)
print(f"\nE_MJ: {len(poly_MJ_large)} terms, constant = {const_MJ_large}")
print(f"E1:   {len(poly_E1_large)} terms, constant = {const_E1_large}")
print(f"E2:   {len(poly_E2_large)} terms, constant = {const_E2_large}")
print(f"E3:   {len(poly_E3_large)} terms, constant = {const_E3_large}")
print(f"\nTotal terms: {len(poly_MJ_large) + len(poly_E1_large) + len(poly_E2_large) + len(poly_E3_large)}")
print(f"Total constant: {offset_total_large}")
print(f"\nQUBO matrix shape: {Q_total_large.shape}")
print(f"Non-zero entries: {np.count_nonzero(Q_total_large)}")
print(f"Sparsity: {100 * (1 - np.count_nonzero(Q_total_large) / Q_total_large.size):.1f}%")

visualize_qubo_matrix(Q_total_large, "QUBO Matrix for HHCHPC", show_values=False)

In [ ]:
# Test configuration for larger example
bitstring_large = '100000000010010000001000000100000001'

print("="*70)
print("Test Configuration for HHCHPC")
print("="*70)
print(f"Bitstring: {bitstring_large}")
print()

print_configuration(bitstring_large, chain_large, num_positions_large)
print()

# Compute energy
total_E_large, breakdown_large = total_energy(chain_large, bitstring_large, adj_large, C, L1, L2, L3, verbose=False)

print(f"\nEnergy Breakdown:")
print(f"  E_MJ (interaction): {breakdown_large[0]}")
print(f"  E1 (one pos/res):   {breakdown_large[1]}  {'✗ VIOLATION' if breakdown_large[1] > 0 else '✓'}")
print(f"  E2 (self-avoid):    {breakdown_large[2]}  {'✗ VIOLATION' if breakdown_large[2] > 0 else '✓'}")
print(f"  E3 (connectivity):  {breakdown_large[3]}  {'✗ VIOLATION' if breakdown_large[3] > 0 else '✓'}")
print(f"\nTotal Energy: {total_E_large}")
print(f"Valid conformation: {is_valid_conformation(breakdown_large)}")

# Verify QUBO
E_qubo_large = evaluate_qubo_energy(bitstring_large, Q_total_large, offset_total_large)
print(f"\nQUBO verification:")
print(f"  Energy via QUBO:  {E_qubo_large:.1f}")
print(f"  Energy via calc:  {total_E_large:.1f}")
print(f"  Match: {'✓' if abs(E_qubo_large - total_E_large) < 0.01 else '✗'}")

## 7. Analysis and Discussion

### Key Observations

1. **Matrix Structure**:
   - The QUBO matrix is sparse (typically 30-50% zeros)
   - Diagonal elements encode linear penalties
   - Off-diagonal elements encode interactions and constraint cross-terms

2. **Energy Landscape**:
   - Valid configurations have E1 = E2 = E3 = 0
   - Invalid configurations are heavily penalized by constraint terms
   - The Lagrange multipliers control the trade-off between constraints and optimization

3. **Scalability**:
   - For N residues and M positions: N×M binary variables
   - QUBO matrix size: (N×M) × (N×M)
   - Number of terms grows as O(N²M²)

4. **Dimensionality**
  
   - Since the graph uses an adj matrix to represent locations, the structure works for any dimensionality
   - Any auxilary information (grid dimensions, node positioning) would just assist for visualization

### Complexity Analysis

In [ ]:
# Analyze scaling
sizes = [(4, 4), (6, 6), (8, 8), (10, 10)]
complexities = []

for n_res, n_pos in sizes:
    n_bits = n_res * n_pos
    matrix_size = n_bits ** 2

    # Estimate number of non-zero entries
    # E_MJ: ~8 * (N choose 2) non-consecutive pairs
    # E1: N * (M choose 2) + N diagonal
    # E2: M * (N choose 2)
    # E3: ~(N-1) * non-adjacent position pairs

    mj_terms = 8 * (n_res * (n_res - 1) // 2)
    e1_terms = n_res * (n_pos * (n_pos - 1) // 2 + 1)
    e2_terms = n_pos * (n_res * (n_res - 1) // 2)
    e3_terms = (n_res - 1) * (n_pos * (n_pos - 1) // 2)

    total_terms = mj_terms + e1_terms + e2_terms + e3_terms

    complexities.append({
        'n_res': n_res,
        'n_pos': n_pos,
        'n_bits': n_bits,
        'matrix_size': matrix_size,
        'total_terms': total_terms,
        'sparsity': 100 * (1 - total_terms / matrix_size)
    })

print("="*70)
print("Complexity Scaling Analysis")
print("="*70)
print(f"\n{'Residues':<10} {'Positions':<10} {'Bits':<8} {'Matrix':<12} {'Terms':<10} {'Sparsity':<10}")
print("-" * 70)
for c in complexities:
    print(f"{c['n_res']:<10} {c['n_pos']:<10} {c['n_bits']:<8} {c['matrix_size']:<12} {c['total_terms']:<10} {c['sparsity']:.1f}%")

# Plot scaling
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

bits = [c['n_bits'] for c in complexities]
terms = [c['total_terms'] for c in complexities]
matrix_sizes = [c['matrix_size'] for c in complexities]

ax1.plot(bits, terms, 'o-', linewidth=2, markersize=8, label='Number of terms')
ax1.set_xlabel('Number of binary variables', fontsize=12)
ax1.set_ylabel('Number of QUBO terms', fontsize=12)
ax1.set_title('QUBO Term Scaling', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.semilogy(bits, matrix_sizes, 'o-', linewidth=2, markersize=8, label='Matrix size', color='red')
ax2.semilogy(bits, terms, 's-', linewidth=2, markersize=8, label='Non-zero terms', color='blue')
ax2.set_xlabel('Number of binary variables', fontsize=12)
ax2.set_ylabel('Count (log scale)', fontsize=12)
ax2.set_title('Matrix Size vs Non-zero Terms', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

### Solving the QUBO

This QUBO formulation can be solved using:

1. **Quantum Annealing** (e.g., D-Wave quantum computers)
   - Directly implements QUBO/Ising model
   - Good for finding approximate solutions quickly

2. **Simulated Annealing** (classical)
   - Probabilistic optimization method
   - Can escape local minima

3. **QAOA** (Quantum Approximate Optimization Algorithm)
   - Hybrid quantum-classical algorithm
   - Runs on gate-based quantum computers

4. **Classical Solvers**
   - Branch and bound
   - Integer programming
   - Genetic algorithms

## Conclusion

This notebook demonstrated:

1. **Problem Formulation**: How to encode protein folding as a QUBO problem
2. **Mathematical Rigor**: Complete derivation of all energy terms
3. **Implementation**: Tested functions for QUBO matrix construction
4. **Validation**: Verification against direct energy calculation
5. **Scalability**: Analysis of how the problem grows with chain length

### Key Takeaways

- The QUBO formulation transforms a constrained optimization problem into an unconstrained one
- Physical constraints are encoded as penalty terms in the objective function
- The resulting QUBO can be solved using quantum or classical optimization methods
- The QUBO matches the stated energy equation; physical minima require sufficiently strong penalties

### Future Directions

- Implement solvers (QAOA, simulated annealing, quantum annealing)
- Incorporate more sophisticated interaction models
- Benchmark against classical protein folding algorithms

---

**All code in this notebook uses tested functions from:**
- `qubo_generation.py` (QUBO matrix construction)
- `calc_mods.py` (Energy evaluation)
- `construction_test.py` (Validation suite)

**Scope:** the example checks and exhaustive small-state regression tests verify the encoded equations for the tested models.

---
# 8. 48-Residue Illustrative QUBO and Simulated Annealing

This section uses the six sequences and 4×4×3 lattice dimensions from Irbäck, Knuthson, and Mohanty (2025), but the supplied 20-type contact coefficients do not match the table cited by that paper. The resulting energies and annealing outcomes are for this repository's contact model. They cannot be compared numerically with the paper's published ground states.


## 8.1 Setup and Installation

In [ ]:
from importlib.util import find_spec
missing = [name for name in ('numpy', 'matplotlib', 'pandas', 'dimod', 'neal')
           if find_spec(name) is None]
if missing:
    raise ImportError(f"Missing {missing}. Install the repository requirements.txt before running this notebook.")


In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt
from collections import defaultdict
from itertools import product
import time

# D-Wave imports
from dimod import BinaryQuadraticModel
from neal import SimulatedAnnealingSampler

print("✓ Imports successful")

## 8.2 Supplied Contact Matrix

The 20×20 matrix below contains signed contact energies (negative is favorable). Its entries differ from at least the Cys–Cys entry in Table VI of Miyazawa and Jernigan (1985), so it should be treated as a supplied illustrative potential rather than the paper's parameterization.


In [ ]:
# One-letter amino acid codes
AMINO_ACIDS = ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I',
               'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']

AA_TO_IDX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

# Supplied symmetric signed contact energies; provenance and units are unverified.
MJ_MATRIX_RAW = np.array([
    # A      R      N      D      C      Q      E      G      H      I      L      K      M      F      P      S      T      W      Y      V
    [-0.20, -0.34, -0.44, -0.42, -0.62, -0.40, -0.42, -0.29, -0.51, -0.73, -0.65, -0.36, -0.64, -0.69, -0.31, -0.34, -0.40, -0.61, -0.53, -0.68],  # A
    [-0.34, -0.15, -0.44, -0.90, -0.55, -0.50, -0.93, -0.40, -0.33, -0.42, -0.45, -0.35, -0.34, -0.41, -0.24, -0.37, -0.32, -0.26, -0.29, -0.37],  # R
    [-0.44, -0.44, -0.28, -0.46, -0.58, -0.44, -0.51, -0.46, -0.54, -0.51, -0.47, -0.46, -0.41, -0.47, -0.34, -0.48, -0.50, -0.36, -0.42, -0.46],  # N
    [-0.42, -0.90, -0.46, -0.19, -0.52, -0.53, -0.49, -0.44, -0.49, -0.42, -0.35, -0.62, -0.33, -0.36, -0.31, -0.48, -0.44, -0.23, -0.30, -0.36],  # D
    [-0.62, -0.55, -0.58, -0.52, -1.54, -0.65, -0.55, -0.54, -0.73, -0.92, -0.88, -0.47, -0.92, -1.01, -0.49, -0.56, -0.61, -0.84, -0.79, -0.91],  # C
    [-0.40, -0.50, -0.44, -0.53, -0.65, -0.29, -0.55, -0.40, -0.49, -0.56, -0.51, -0.46, -0.47, -0.54, -0.31, -0.43, -0.41, -0.39, -0.42, -0.51],  # Q
    [-0.42, -0.93, -0.51, -0.49, -0.55, -0.55, -0.22, -0.43, -0.42, -0.44, -0.37, -0.64, -0.35, -0.40, -0.30, -0.47, -0.43, -0.21, -0.27, -0.38],  # E
    [-0.29, -0.40, -0.46, -0.44, -0.54, -0.40, -0.43, -0.14, -0.46, -0.52, -0.46, -0.41, -0.47, -0.51, -0.26, -0.37, -0.39, -0.44, -0.44, -0.47],  # G
    [-0.51, -0.33, -0.54, -0.49, -0.73, -0.49, -0.42, -0.46, -0.34, -0.66, -0.62, -0.35, -0.64, -0.72, -0.37, -0.46, -0.47, -0.62, -0.57, -0.60],  # H
    [-0.73, -0.42, -0.51, -0.42, -0.92, -0.56, -0.44, -0.52, -0.66, -1.00, -0.95, -0.41, -0.96, -1.04, -0.49, -0.52, -0.59, -0.90, -0.83, -0.98],  # I
    [-0.65, -0.45, -0.47, -0.35, -0.88, -0.51, -0.37, -0.46, -0.62, -0.95, -0.91, -0.37, -0.93, -1.01, -0.46, -0.47, -0.53, -0.89, -0.80, -0.93],  # L
    [-0.36, -0.35, -0.46, -0.62, -0.47, -0.46, -0.64, -0.41, -0.35, -0.41, -0.37, -0.18, -0.32, -0.39, -0.23, -0.39, -0.36, -0.19, -0.24, -0.35],  # K
    [-0.64, -0.34, -0.41, -0.33, -0.92, -0.47, -0.35, -0.47, -0.64, -0.96, -0.93, -0.32, -0.89, -0.98, -0.45, -0.47, -0.52, -0.86, -0.78, -0.92],  # M
    [-0.69, -0.41, -0.47, -0.36, -1.01, -0.54, -0.40, -0.51, -0.72, -1.04, -1.01, -0.39, -0.98, -1.10, -0.50, -0.51, -0.57, -0.98, -0.89, -1.02],  # F
    [-0.31, -0.24, -0.34, -0.31, -0.49, -0.31, -0.30, -0.26, -0.37, -0.49, -0.46, -0.23, -0.45, -0.50, -0.16, -0.30, -0.32, -0.42, -0.38, -0.45],  # P
    [-0.34, -0.37, -0.48, -0.48, -0.56, -0.43, -0.47, -0.37, -0.46, -0.52, -0.47, -0.39, -0.47, -0.51, -0.30, -0.33, -0.41, -0.41, -0.41, -0.48],  # S
    [-0.40, -0.32, -0.50, -0.44, -0.61, -0.41, -0.43, -0.39, -0.47, -0.59, -0.53, -0.36, -0.52, -0.57, -0.32, -0.41, -0.38, -0.45, -0.45, -0.55],  # T
    [-0.61, -0.26, -0.36, -0.23, -0.84, -0.39, -0.21, -0.44, -0.62, -0.90, -0.89, -0.19, -0.86, -0.98, -0.42, -0.41, -0.45, -0.86, -0.76, -0.84],  # W
    [-0.53, -0.29, -0.42, -0.30, -0.79, -0.42, -0.27, -0.44, -0.57, -0.83, -0.80, -0.24, -0.78, -0.89, -0.38, -0.41, -0.45, -0.76, -0.69, -0.78],  # Y
    [-0.68, -0.37, -0.46, -0.36, -0.91, -0.51, -0.38, -0.47, -0.60, -0.98, -0.93, -0.35, -0.92, -1.02, -0.45, -0.48, -0.55, -0.84, -0.78, -0.96],  # V
])

def get_mj_energy(aa1, aa2):
    """Get the supplied signed contact energy for two amino acids."""
    i, j = AA_TO_IDX[aa1], AA_TO_IDX[aa2]
    return MJ_MATRIX_RAW[i, j]

print("Supplied Contact Matrix:")
print(f"  Dimensions: {MJ_MATRIX_RAW.shape}")
print(f"  Amino acids: {len(AMINO_ACIDS)}")
print(f"  Range: [{MJ_MATRIX_RAW.min():.2f}, {MJ_MATRIX_RAW.max():.2f}]")
print(f"\nExample interactions (negative = favorable):")
print(f"  C-C (Cys-Cys): {get_mj_energy('C', 'C'):.2f} (strongest attraction)")
print(f"  F-F (Phe-Phe): {get_mj_energy('F', 'F'):.2f} (hydrophobic)")
print(f"  K-K (Lys-Lys): {get_mj_energy('K', 'K'):.2f} (weakest)")
print(f"  G-G (Gly-Gly): {get_mj_energy('G', 'G'):.2f} (small residue)")

## 8.3 Sequences from Irbäck et al.

The sequence strings and published 3D minimum energies below come from Table I. The published energies are metadata for the paper's model and are not targets for this notebook's supplied contact matrix.


In [ ]:
# Sequences from Table I of Irbäck et al. (2025)
# Each sequence folds to topology A (low complexity) or B (high complexity)

SEQUENCES = {
    1: {
        'sequence': 'FRTRPLNHDFYNYKIWEPFKPADFPKAWDRMLDHVWDSMASWGHQHCS',
        'topology': 'A',
        'E_min': -25.85
    },
    2: {
        'sequence': 'CDLPPFTYRHHGNDFWKNYEMIKHWDLWRDMFRAFWSDPVKASPHQAS',
        'topology': 'A',
        'E_min': -25.92
    },
    3: {
        'sequence': 'FRTPWVSHQFYAYKLMEHFKWGDFCRNMDKWIDSLPDRWNPAPHDHAS',
        'topology': 'A',
        'E_min': -26.09
    },
    4: {
        'sequence': 'KDKIHFRMNYGYPAWDAQSVKDLTCPRDWHFPHMRDPSHNWELAFFWS',
        'topology': 'B',
        'E_min': -25.87
    },
    5: {
        'sequence': 'ENDVTMDMDPSPCLFRIHNLPRAHSFDRFGWHQFDKYHYKWKWAWAPS',
        'topology': 'B',
        'E_min': -26.15
    },
    6: {
        'sequence': 'EHDAQLDFDWSRWTWHGRNSYHAPAMYRWPVHDMDKPNPKFKIFFLCS',
        'topology': 'B',
        'E_min': -26.24
    }
}

print("Sequences from Irbäck et al. (2025):")
print("="*70)
for seq_id, data in SEQUENCES.items():
    print(f"\nSequence {seq_id} (Topology {data['topology']}):")
    print(f"  {data['sequence'][:24]}")
    print(f"  {data['sequence'][24:]}")
    print(f"  Length: {len(data['sequence'])}, E_min: {data['E_min']:.2f}")

## 8.4 Building the 4×4×3 Cubic Lattice

The lattice has 48 positions arranged in a 4×4×3 grid. Two positions are adjacent if they differ by 1 in exactly one coordinate.

In [ ]:
def build_cubic_lattice(dims):
    """
    Build adjacency matrix for a cubic lattice.
    
    Args:
        dims: tuple (Lx, Ly, Lz) giving lattice dimensions
    
    Returns:
        adj: adjacency matrix (num_sites x num_sites)
        coords: list of (x, y, z) coordinates for each site
    """
    Lx, Ly, Lz = dims
    num_sites = Lx * Ly * Lz
    
    # Create coordinate list
    coords = []
    for z in range(Lz):
        for y in range(Ly):
            for x in range(Lx):
                coords.append((x, y, z))
    
    # Build adjacency matrix
    adj = np.zeros((num_sites, num_sites), dtype=int)
    
    for i, (x1, y1, z1) in enumerate(coords):
        for j, (x2, y2, z2) in enumerate(coords):
            # Manhattan distance = 1 means adjacent
            dist = abs(x1-x2) + abs(y1-y2) + abs(z1-z2)
            if dist == 1:
                adj[i, j] = 1
    
    return adj, coords

# Build 4x4x3 lattice
LATTICE_DIMS = (4, 4, 3)
adj_matrix, lattice_coords = build_cubic_lattice(LATTICE_DIMS)
NUM_POSITIONS = len(lattice_coords)

print(f"4×4×3 Cubic Lattice:")
print(f"  Dimensions: {LATTICE_DIMS}")
print(f"  Number of sites: {NUM_POSITIONS}")
print(f"  Total edges (adjacencies): {np.sum(adj_matrix) // 2}")
print(f"  Average coordination number: {np.sum(adj_matrix) / NUM_POSITIONS:.2f}")

## 8.5 QUBO Construction

The same assignment and lattice constraints as equations (2)–(6) of the paper are used. Contact coefficients are taken from the supplied signed matrix, and the paper's penalty vector is used only as an illustrative parameter choice. These weights have not been proven sufficient for this different potential.


In [ ]:
def bit_index(residue, position, num_positions):
    """Convert (residue, position) to bit index."""
    return residue * num_positions + position


def build_protein_qubo(sequence, adj, lambda1=1.5, lambda2=2.0, lambda3=2.0):
    """
    Build complete QUBO for protein folding.
    
    Args:
        sequence: string of amino acid letters
        adj: adjacency matrix for lattice
        lambda1, lambda2, lambda3: Lagrange multipliers
    
    Returns:
        linear: dict of linear biases
        quadratic: dict of quadratic biases
        offset: constant energy offset
    """
    N = len(sequence)
    M = adj.shape[0]
    num_bits = N * M
    
    linear = defaultdict(float)
    quadratic = defaultdict(float)
    offset = 0.0
    
    # ================================================================
    # Contact energy: the supplied matrix already stores signed energies.
    # The paper writes -C; its positive contact strengths would need to be
    # negated before comparing them with these signed coefficients.
    # ================================================================
    for i in range(N):
        for j in range(i + 2, N):  # |i-j| > 1 (non-consecutive)
            # Negative supplied coefficients favor contacts.
            C_ij = MJ_MATRIX_RAW[AA_TO_IDX[sequence[i]], AA_TO_IDX[sequence[j]]]
            
            # Sum over adjacent position pairs
            for n in range(M):
                for m in range(M):
                    if adj[n, m] == 1:
                        bit_i = bit_index(i, n, M)
                        bit_j = bit_index(j, m, M)
                        
                        # Add the signed pair energy once.
                        key = (min(bit_i, bit_j), max(bit_i, bit_j))
                        quadratic[key] += C_ij
    
    # ================================================================
    # E1: One position per residue (Eq. 4)
    # E1 = sum_i (sum_n b_in - 1)^2
    #    = sum_i [sum_n b_in^2 - 2*sum_n b_in + 2*sum_{n<m} b_in*b_im + 1]
    #    = sum_i [-sum_n b_in + 2*sum_{n<m} b_in*b_im + 1]  (since b^2 = b)
    # ================================================================
    for i in range(N):
        # Linear terms: -1 per bit
        for n in range(M):
            bit = bit_index(i, n, M)
            linear[bit] += lambda1 * (-1)
        
        # Quadratic terms: +2 for each pair within same residue
        for n in range(M):
            for m in range(n + 1, M):
                bit_n = bit_index(i, n, M)
                bit_m = bit_index(i, m, M)
                key = (bit_n, bit_m)
                quadratic[key] += lambda1 * 2
        
        # Constant: +1 per residue
        offset += lambda1 * 1
    
    # ================================================================
    # E2: Self-avoidance / one residue per position (Eq. 5)
    # E2 = (1/2) * sum_n sum_{i!=j} b_in * b_jn
    #    = sum_n sum_{i<j} b_in * b_jn
    # ================================================================
    for n in range(M):
        for i in range(N):
            for j in range(i + 1, N):
                bit_i = bit_index(i, n, M)
                bit_j = bit_index(j, n, M)
                key = (bit_i, bit_j)
                quadratic[key] += lambda2 * 1
    
    # ================================================================
    # E3: Chain connectivity (Eq. 6)
    # E3 = sum_{i<N} sum_n b_in * sum_{||m-n||>1} b_{i+1,m}
    # Penalizes consecutive residues at non-adjacent positions
    # ================================================================
    # Build non-adjacency matrix (excluding self)
    non_adj = 1 - adj - np.eye(M)
    
    for i in range(N - 1):
        for n in range(M):
            for m in range(M):
                if non_adj[n, m] == 1:
                    bit_i = bit_index(i, n, M)
                    bit_j = bit_index(i + 1, m, M)
                    key = (min(bit_i, bit_j), max(bit_i, bit_j))
                    quadratic[key] += lambda3 * 1
    
    return dict(linear), dict(quadratic), offset


print("✓ QUBO construction functions defined")

In [ ]:
# Build QUBO for sequence 5 (used as main example in paper)
seq_id = 5
sequence = SEQUENCES[seq_id]['sequence']
paper_E_min = SEQUENCES[seq_id]['E_min']

print(f"Building QUBO for Sequence {seq_id}:")
print(f"  Sequence: {sequence[:24]}...")
print(f"  Length: {len(sequence)}")
print(f"  Paper E_min (different contact model): {paper_E_min}")
print()

# Use paper's Lagrange multipliers
LAMBDA1, LAMBDA2, LAMBDA3 = 1.5, 2.0, 2.0

start_time = time.time()
linear, quadratic, offset = build_protein_qubo(
    sequence, adj_matrix, 
    lambda1=LAMBDA1, lambda2=LAMBDA2, lambda3=LAMBDA3
)
build_time = time.time() - start_time

num_bits = len(sequence) * NUM_POSITIONS

print(f"QUBO Statistics:")
print(f"  Binary variables: {num_bits}")
print(f"  Linear terms: {len(linear)}")
print(f"  Quadratic terms: {len(quadratic)}")
print(f"  Offset: {offset:.2f}")
print(f"  Lagrange multipliers: λ1={LAMBDA1}, λ2={LAMBDA2}, λ3={LAMBDA3}")
print(f"  Build time: {build_time:.2f} s")

## 8.6 Simulated Annealing Solver

The cited paper uses 25 temperatures with sweeps at each temperature. The sampler below uses its own schedule and a total sweep count per read. The numerical settings and run times are therefore not direct replications of the paper's SA protocol.


In [ ]:
def solve_with_sa(linear, quadratic, offset, num_reads=10, num_sweeps=10000, 
                  beta_range=None, seed=None):
    """
    Solve QUBO using D-Wave's neal Simulated Annealing.
    
    Args:
        linear: dict of linear biases
        quadratic: dict of quadratic biases  
        offset: constant offset
        num_reads: number of independent runs
        num_sweeps: sweeps per run
        beta_range: (beta_min, beta_max) for annealing schedule
        seed: random seed
    
    Returns:
        result: SampleSet from neal
    """
    # Create BQM
    bqm = BinaryQuadraticModel(linear, quadratic, offset, vartype='BINARY')
    
    # Create sampler
    sampler = SimulatedAnnealingSampler()
    
    # Set default beta_range if not provided
    # Paper uses beta_i = 1.05^i for i=1..25, giving range ~1.05 to ~3.4
    if beta_range is None:
        beta_range = (1.0, 50.0)  # Wider range for neal's schedule
    
    # Run SA
    result = sampler.sample(
        bqm,
        num_reads=num_reads,
        num_sweeps=num_sweeps,
        beta_range=beta_range,
        seed=seed
    )
    
    return result

print("✓ SA solver function defined")

In [ ]:
def validate_solution(sample, sequence, adj, num_positions):
    """
    Validate a solution and compute energy components.
    
    Returns:
        is_valid: bool
        E_MJ: interaction energy (the physics we care about)
        violations: dict with E1, E2, E3 penalty counts
    """
    N = len(sequence)
    M = num_positions
    
    if M != adj.shape[0] or set(sample) != set(range(N * M)) or any(value not in (0, 1) for value in sample.values()):
        raise ValueError('sample must contain exactly N*M binary variables for the lattice')

    # Convert sample to bit matrix b[i,n]
    b = np.zeros((N, M))
    for bit_idx, val in sample.items():
        if val == 1:
            i = bit_idx // M
            n = bit_idx % M
            if i < N:  # Safety check
                b[i, n] = 1
    
    # E1: Each residue at exactly one position
    # Penalty = (sum_n b_in - 1)^2 for each residue
    E1 = np.sum((np.sum(b, axis=1) - 1) ** 2)
    
    # E2: Each position has at most one residue
    # Penalty = sum over pairs at same position
    E2 = 0
    for n in range(M):
        occupancy = np.sum(b[:, n])
        E2 += occupancy * (occupancy - 1) / 2
    
    # E3: Chain connectivity
    # Penalty for consecutive residues at non-adjacent positions
    non_adj = 1 - adj - np.eye(M)
    E3 = 0
    for i in range(N - 1):
        E3 += b[i, :] @ non_adj @ b[i+1, :]
    
    # Contact energy from the supplied signed matrix
    E_MJ = 0
    for i in range(N):
        for j in range(i + 2, N):
            C_ij = MJ_MATRIX_RAW[AA_TO_IDX[sequence[i]], AA_TO_IDX[sequence[j]]]
            for n in range(M):
                for m in range(M):
                    if adj[n, m] == 1:
                        E_MJ += C_ij * b[i, n] * b[j, m]
    
    is_valid = (E1 == 0) and (E2 == 0) and (E3 == 0)
    
    return is_valid, E_MJ, {'E1': E1, 'E2': E2, 'E3': E3}


def get_residue_positions(sample, sequence, num_positions):
    """Extract residue positions from solution."""
    N = len(sequence)
    M = num_positions
    positions = {}
    
    for bit_idx, val in sample.items():
        if val == 1:
            i = bit_idx // M
            n = bit_idx % M
            if i not in positions:
                positions[i] = []
            positions[i].append(n)
    
    return positions

print("✓ Validation functions defined")

## 8.7 Running Simulated Annealing

Let's run SA on sequence 5 with parameters closer to the paper.

In [ ]:
# Test with moderate settings first
print("="*70)
print(f"SIMULATED ANNEALING: Sequence {seq_id}")
print("No exact minimum is known for this supplied 48-residue contact model.")
print(f"Lagrange multipliers: λ1={LAMBDA1}, λ2={LAMBDA2}, λ3={LAMBDA3}")
print("="*70)

# Test different sweep counts
sweep_tests = [5000, 10000, 25000, 50000]
results_summary = []

for num_sweeps in sweep_tests:
    print(f"\n--- Testing {num_sweeps} sweeps ---")
    
    start_time = time.time()
    result = solve_with_sa(
        linear, quadratic, offset,
        num_reads=20,
        num_sweeps=num_sweeps,
        beta_range=(1.0, 100.0),
        seed=42
    )
    solve_time = time.time() - start_time
    
    # Analyze results
    best = result.first
    is_valid, E_MJ, violations = validate_solution(
        best.sample, sequence, adj_matrix, NUM_POSITIONS
    )
    
    # Count valid solutions and find best valid E_MJ
    valid_count = 0
    best_valid_E_MJ = float('inf')
    for datum in result.data():
        v, e_mj, penalty = validate_solution(datum.sample, sequence, adj_matrix, NUM_POSITIONS)
        direct_total = e_mj + LAMBDA1*penalty['E1'] + LAMBDA2*penalty['E2'] + LAMBDA3*penalty['E3']
        if not np.isclose(datum.energy, direct_total, rtol=0, atol=1e-6):
            raise AssertionError('Sampler QUBO energy disagrees with direct evaluation')
        if v:
            valid_count += 1
            best_valid_E_MJ = min(best_valid_E_MJ, e_mj)
    
    results_summary.append({
        'sweeps': num_sweeps,
        'time': solve_time,
        'best_energy': best.energy,
        'best_E_MJ': best_valid_E_MJ if valid_count else None,
        'valid_count': valid_count,
        'is_valid': is_valid,
        'violations': violations
    })
    
    print(f"  Time: {solve_time:.2f} s")
    print(f"  Best QUBO energy: {best.energy:.4f}")
    print(f"  Valid solutions: {valid_count}/20")
    if valid_count > 0:
        print(f"  Best valid contact energy: {best_valid_E_MJ:.4f}")
    else:
        print(f"  No valid solutions found")
        print(f"  Best solution violations: E1={violations['E1']:.0f}, E2={violations['E2']:.0f}, E3={violations['E3']:.0f}")

In [ ]:
# Longer neal run; this is not 80,000 sweeps at each of 25 temperatures.
print("\n" + "="*70)
print("EXTENDED NEAL SA RUN (80,000 total sweeps per read)")
print("="*70)

start_time = time.time()
result_long = solve_with_sa(
    linear, quadratic, offset,
    num_reads=50,
    num_sweeps=80000,
    beta_range=(1.0, 100.0),
    seed=123
)
long_solve_time = time.time() - start_time

# Find best valid solution
best_valid = None
best_valid_E_MJ = float('inf')
valid_count = 0
all_E_MJ = []

for datum in result_long.data():
    is_valid, E_MJ, violations = validate_solution(
        datum.sample, sequence, adj_matrix, NUM_POSITIONS
    )
    direct_total = E_MJ + LAMBDA1*violations['E1'] + LAMBDA2*violations['E2'] + LAMBDA3*violations['E3']
    if not np.isclose(datum.energy, direct_total, rtol=0, atol=1e-6):
        raise AssertionError('Sampler QUBO energy disagrees with direct evaluation')
    if is_valid:
        valid_count += 1
        all_E_MJ.append(E_MJ)
        if E_MJ < best_valid_E_MJ:
            best_valid_E_MJ = E_MJ
            best_valid = datum

print(f"\nResults:")
print(f"  Total time: {long_solve_time:.2f} s")
print(f"  Valid solutions: {valid_count}/50")

if valid_count > 0:
    print(f"  Best valid E_MJ: {best_valid_E_MJ:.4f}")
    print(f"  Mean valid contact energy: {np.mean(all_E_MJ):.4f}")
else:
    print(f"  No valid solutions found.")
    # Show best invalid solution's violations
    best = result_long.first
    _, _, violations = validate_solution(best.sample, sequence, adj_matrix, NUM_POSITIONS)
    print(f"  Best solution violations: E1={violations['E1']:.0f}, E2={violations['E2']:.0f}, E3={violations['E3']:.0f}")

## 8.8 Discussion: Why Finding Valid Solutions is Hard

If SA is not finding valid solutions, there are several possible reasons:

1. **Lagrange multipliers too weak**: The constraints aren't penalized enough relative to the MJ energy gains
2. **Annealing schedule**: neal's default schedule may differ from the paper's custom implementation
3. **Problem difficulty**: The paper notes this is a hard problem - even their SA took ~2000s with 80,000 sweeps

The paper's key insight is that **D-Wave's hybrid quantum-classical solver** is much more effective, finding solutions in ~10 seconds with 100% success rate.

In [ ]:
# These are feasibility measurements for this notebook's supplied contact model.
import pandas as pd
df = pd.DataFrame(results_summary)
print(df[['sweeps', 'time', 'best_energy', 'best_E_MJ', 'valid_count', 'is_valid']].to_string(index=False))
print(f"Extended run: {valid_count}/50 valid samples; best valid contact energy: "
      f"{best_valid_E_MJ:.4f}" if valid_count else "Extended run: no valid samples.")
print("Validity is not the same as reaching the unknown physical optimum.")


---
# 9. State-Vector Simulation Size

The following calculation estimates memory for a dense state-vector simulation of the assignment-bit circuit. It does not establish a general limit on QAOA or on quantum hardware.


In [ ]:
print("Dense state-vector memory estimate (16 bytes per complex amplitude)")
for label, residues, sites in [
    ("Toy", 4, 4), ("Small", 6, 6), ("Medium", 16, 16),
    ("Paper dimensions", 48, 48),
]:
    qubits = residues * sites
    log10_bytes = qubits * math.log10(2) + math.log10(16)
    print(f"{label}: {qubits} assignment qubits, approximately 10^{log10_bytes:.1f} bytes")
print("This estimate concerns direct dense simulation only.")


---
# 10. Conclusions

The tutorial's QUBO equations encode the stated lattice contact and constraint model. The toy examples and regression tests check their coefficients and feasibility conditions. The supplied 20-type matrix does not match the contact potential cited by the paper, so this notebook's 48-residue energies and annealing outcomes must not be interpreted as replications of its numerical results.

For each annealing run, report feasibility separately from best valid contact energy. The exact minimum for the supplied 48-residue model is unknown here.
